# CPT Smoke Test — Base gốc vs Checkpoint hiện tại (§6.9)

So sánh nhanh **raw `Qwen/Qwen3-1.7B-Base`** với **checkpoint CPT mới nhất trên Hub**, trên cùng `eval_vi` split (packed 2048-token, tách riêng từ Notebook A, model chưa từng thấy). Không train — chỉ load model 2 lần, tính loss trung bình + sinh thử văn bản. ~15-20 phút GPU T4, không đụng tới ngân sách/checkpoint của Notebook B.

Dùng khi muốn biết CPT có tác dụng thật hay chưa mà không cần đợi W&B curve rõ xu hướng — loss thấp hơn + văn bản mạch lạc hơn ở checkpoint so với base là bằng chứng trực tiếp.

**Trước khi chạy:** bật GPU T4; Add-ons → Secrets → tick `HF_TOKEN` cho notebook này (read token cũng đủ, không cần ghi gì lên Hub).

In [ ]:
!pip install -q -U unsloth

In [ ]:
# Config + login — username tự lấy từ token, không gõ tay để khỏi sai namespace
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami, HfApi, snapshot_download
login(UserSecretsClient().get_secret("HF_TOKEN"))

HF_USER = whoami()["name"]
print(f"HF user: {HF_USER}")

DATA_REPO = f"{HF_USER}/vi-cpt-corpus-2048"
CKPT_REPO = f"{HF_USER}/Qwen3-1.7B-vi-cpt-ckpt"
N_EVAL    = 100   # số block 2048-token lấy từ eval_vi để tính loss trung bình — đủ ổn định, đủ nhanh
PROMPT    = "Việt Nam là một quốc gia"

In [ ]:
# Lấy subset eval_vi (model chưa từng thấy — tách riêng ở Notebook A)
from datasets import load_dataset
eval_vi = load_dataset(DATA_REPO, split="eval_vi").select(range(N_EVAL))
print(f"Lấy {len(eval_vi)} block từ eval_vi để so sánh")

In [ ]:
# Loss trung bình trên eval_vi — mỗi block đã đúng 2048 token nên không cần padding/mask
import torch

def eval_loss(model, ds, batch_size=2):
    model.eval()
    losses = []
    with torch.no_grad():
        for i in range(0, len(ds), batch_size):
            ids = torch.tensor(ds[i:i + batch_size]["input_ids"]).to(model.device)
            losses.append(model(input_ids=ids, labels=ids).loss.item())
    return sum(losses) / len(losses)

In [ ]:
# 1) Raw base — chưa CPT gì cả
from unsloth import FastLanguageModel
base_model, base_tok = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3-1.7B-Base", max_seq_length=2048, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(base_model)

base_loss = eval_loss(base_model, eval_vi)
print(f"Raw base — eval_vi loss: {base_loss:.4f}")

inputs = base_tok(PROMPT, return_tensors="pt").to(base_model.device)
base_gen = base_tok.decode(
    base_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)[0],
    skip_special_tokens=True)
print("Base sinh:", base_gen)

In [ ]:
# Giải phóng VRAM trước khi load model thứ 2 (T4 16GB không đủ giữ cả 2 cùng lúc)
import gc
del base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 2) Checkpoint CPT mới nhất trên Hub (tự tìm step lớn nhất — không cần biết trước là bao nhiêu)
api = HfApi()
steps = sorted(int(f.split("/")[0].split("-")[1]) for f in api.list_repo_files(CKPT_REPO)
               if f.startswith("checkpoint-"))
assert steps, f"Chưa có checkpoint nào trên {CKPT_REPO}"
ckpt_name = f"checkpoint-{steps[-1]}"
print(f"So sánh với: {ckpt_name}")

local_dir = snapshot_download(CKPT_REPO, allow_patterns=f"{ckpt_name}/*", local_dir="/kaggle/working/smoke-ckpt")
local_ckpt = os.path.join(local_dir, ckpt_name)

In [ ]:
# adapter_config.json trong checkpoint tự trỏ về base model gốc — Unsloth load thẳng base+adapter
ckpt_model, ckpt_tok = FastLanguageModel.from_pretrained(
    local_ckpt, max_seq_length=2048, load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(ckpt_model)

ckpt_loss = eval_loss(ckpt_model, eval_vi)
print(f"{ckpt_name} — eval_vi loss: {ckpt_loss:.4f}")

inputs = ckpt_tok(PROMPT, return_tensors="pt").to(ckpt_model.device)
ckpt_gen = ckpt_tok.decode(
    ckpt_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)[0],
    skip_special_tokens=True)
print(f"{ckpt_name} sinh:", ckpt_gen)

In [ ]:
# So sánh — dương = CPT đang cải thiện thật; ~0 hoặc âm = CPT chưa có tác dụng rõ ở checkpoint này
print(f"\n=== So sánh trên {N_EVAL} block eval_vi ===")
print(f"Raw base:        loss={base_loss:.4f}")
print(f"{ckpt_name:>14}: loss={ckpt_loss:.4f}")
print(f"Chênh lệch (base - ckpt): {base_loss - ckpt_loss:+.4f}")